In [2]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
sys.path.append("../../benchmark")
import test_base
from sentence_splitter import split_text_into_sentences
from BERT_classifier.Classify_report_with_BERT import classification_report_BERT

In [3]:
sentence_length = 6

## Test using a BERT-Classification Model trained on paragraphs to classify an entire report.

**Function:** pdf -> NACE Class

In [4]:
dataset_path = "data/datasets/german_annual_reports"
dataset_path = "data/datasets/stoxx_600_extended"
dataset_path = "data/datasets/reports_subset_from_full_data_1"
dataset_path = "data/datasets/stoxx_600"

In [5]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [6]:
results_path = "results/BERT_classification/dataset__stoxx_600_sentence_len_6__nace_level_1"
results_path = "results/BERT_classification/2_dataset__reports_subset_from_full_data_1_sentence_len_6__nace_level_1"
results_path = "results/BERT_classification/2_dataset__stoxx_600_sentence_len_6__nace_level_1"
reports_results = glob.glob(results_path+ "/*/*.csv")
reports_results

['results/BERT_classification/2_dataset__stoxx_600_sentence_len_6__nace_level_1/Hannover Rueck SE1.txt/Hannover Rueck SE1.txt_classifications.csv']

In [7]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [8]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0, sep=",")
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,SalMar ASA,SALM-NO,SALM-NO,1984.965581,2458.824022,2271.61166349053,3.21,A,salmar-annual-report-2022.pdf
1,Bakkafrost P/F,BAKKA-NO,BAKKA-NO,929.503079,937.862170,973.053513491168,3.21,A,Bakkafrost PF2.pdf
2,Antofagasta plc,ANTO-GB,ANTO-GB,5577.681426,5849.975673,6113.94698310345,7.29,B,Antofagasta plc1.pdf
3,Anglo American plc,AAL-GB,AAL-GB,33423.271144,28355.894415,25288.1884924262,7.29,B,Anglo American plc1.pdf
4,TotalEnergies SE,TTE-FR,TTE-FR,250538.948328,202517.658053,180837.266896225,6.10,B,Totalenergies EP Gabon1.pdf


In [9]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
len(report_to_nace_class)

294

In [10]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
len(reports_path)

263

In [11]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [12]:
%ls results/BERT_models/

finbert__train_full_model__all_labels/
results__all_labels__train_classifier_only/
results__bert-base-uncased__train_full_model__all_labels/
results__bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_045bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_045finbert__train_full_model__all_labels/
results__cos_thresh_05bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_05finbert__train_full_model__all_labels/
results_data_2__cos_thresh_045bert-base-uncased__train_full_model__some_labels/
results_data_2__cos_thresh_045finbert__train_full_model__all_labels/
results_data_2__cos_thresh_05bert-base-uncased__train_full_model__some_labels/
results_data_2__cos_thresh_05finbert__train_full_model__all_labels/
results_data_2__cos_thresh_06bert-base-uncased__train_full_model__some_labels/
results__finbert__train_full_model__all_labels/
results__finbert__train_full_model__some_labels/
results__new_approach_data__num_layers_2bert-base-uncased__t

In [13]:
ckpt = "results/BERT_models/results_null_classifiers__cos_thres_0.5__bert-base-uncased__train_full_model__some_labels/checkpoint-2331"
ckpt = "results/BERT_models/results__new_approach_data__num_layers_2bert-base-uncased__train_full_model__some_labels/checkpoint-15990"
model, tokenizer, device = classification_report_BERT.load_model(ckpt_path=ckpt)

Some weights of the model checkpoint at results/BERT_models/results__new_approach_data__num_layers_2bert-base-uncased__train_full_model__some_labels/checkpoint-15990 were not used when initializing BertForSequenceClassification: ['classifier.0.bias', 'classifier.0.weight', 'classifier.3.bias', 'classifier.3.weight', 'classifier.6.bias', 'classifier.6.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at results/BERT_models/results__new_approach_data__n

In [14]:
import numpy as np
import pandas as pd

# assume df is your dataframe
def softmax(x):
    e = np.exp(x - np.max(x))   # subtract max for numerical stability
    return e / e.sum()

In [15]:
def majority_vote(df: pd.DataFrame, scores: list) -> dict:
    
    df["max_class_sim"] = [scores[i] for i in np.argmax(df[scores], 1)]
    evaluation = (df["max_class_sim"].value_counts() / len(df)).sort_values(ascending=False).to_dict()

    return evaluation

In [16]:
def confidence_weighted_voting(df: pd.DataFrame, scores: list) -> dict:

    df["max_class_sim"] = [scores[i] for i in np.argmax(df[scores], 1)]
    df["max_sim"] = df[scores].max(1)
    evaluation = (df.groupby("max_class_sim")["max_sim"].sum().sort_values(ascending=False)/len(df)).to_dict()

    return evaluation

In [17]:
def logit_stacking(df: pd.DataFrame, scores: list) -> dict:



    return evaluation

In [18]:
#reports_results = ['../../results/BERT_classification/test_1/Heritage Foods Limited1.txt/Heritage Foods Limited1.txt_classifications.csv']

In [19]:
level = 1
recording = [] 

#if path_nace_code_descriptions is None: 
path_nace_code_descriptions = "data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
    
df_nace_codes_descriptions = pd.read_csv(path_nace_code_descriptions, sep="\t")

#for results_path in tqdm.tqdm((reports_results[1:])): 
for results_path in ((reports_results)): 

    report_name = os.path.basename(results_path.split("/")[-2])
    #label = report_to_nace_class.get(os.path.basename(report_path))
    label = report_to_nace_class.get(report_name)

    level_1_label = test_base.get_all_level(label)[1]

    # # retrieve chunks
    # chunks = preprocess_report(pdf_path=report_path)
    # # remove chunks with length smaller than threshold
    # chunks = [chunk for chunk in chunks if len(chunk) > threshold_min_chunk_len]
    
    # # remove duplicates
    # chunks = list(set(chunks))
    
    # if len(chunks) == 0:
    #     continue
    # class_eval_dict = classification_function(
    #     chunks=chunks, 
    #     level=level,
    #     df_nace_codes_descriptions=df_nace_codes_descriptions, 
    #     cos_threshold=cos_threshold, 
    #     report_path=report_path, 
    #     result_path=result_path, 
    #     model = model,
    #     tokenizer = tokenizer,
    #     device = device,
    # )
    # mean_vals = classification_function(
    #     **classification_function_inputs
    # )

    df_eval = pd.read_csv(results_path, index_col=0)
    scores = df_eval.columns.drop("Sentences")
    df_softmax = df_eval[scores].apply(softmax, axis=1)
    eval = confidence_weighted_voting(
        df_softmax, scores
    )

    print(df_softmax[scores].max().max())

    print(eval)

    #df_softmax = df_eval[scores].apply(softmax, axis=1)
    #print(df_softmax.max().max())

    # print(level_1_label)
    # print(eval)
    # # get label of the report
    # label = report_to_nace_class.get(os.path.basename(report_path))
    
    # try: 
    #     if isinstance(label, float) or isinstance(label, int):                     
    #         if len(str(label).split(".")[0]) == 1: 
    #             label = "0" + str(label) 
    #     label_description = df_nace_codes_descriptions[df_nace_codes_descriptions["CODE"] == str(label)]["NAME"].iloc[0]
    #     evaluation = test_base.get_evaluation(class_eval_dict, label, df_nace_codes_descriptions, level)
    # except IndexError: 
    #     label_description = ""
    #     evaluation = {}
    # recording.append({"name": os.path.basename(report_path), "NACE": label, "label_description": label_description, **evaluation})
    # df_recording = pd.DataFrame(recording)
    # df_recording.to_csv(os.path.join(result_path, "recordings.csv"))


0.17359551419063068
{'P': 0.04833255287212852, 'J': 0.04763744544013977, 'N': 0.00963051584754535, 'I': 0.007538353474715317, 'K': 0.002820441867220873, 'A': 0.0017213357987776374, 'L': 0.0016467397608481782, 'D': 0.0013914546461685376, 'C': 0.00033741132631591344, 'NO_CLASS': 0.0003233081112169926, 'H': 0.0003124903676325121, 'F': 0.0002967118249850743, 'Q': 0.0002609776030120413}


In [ ]:
df_eval.max()

F                                                     0.642904
P                                                     0.474345
L                                                     0.743894
K                                                     0.494121
I                                                      0.65594
A                                                     0.895658
D                                                     0.918923
G                                                     0.760219
E                                                     1.044022
NO_CLASS                                              0.953975
B                                                     0.876131
J                                                     0.717586
H                                                     0.475585
Q                                                     0.725816
M                                                     0.822036
N                                                     0

In [ ]:
df_softmax = df_eval[scores].apply(softmax, axis=1)
df_softmax.mean(0)

F           0.054689
P           0.048024
L           0.047983
K           0.025974
I           0.061484
A           0.084762
D           0.074989
G           0.049602
E           0.098610
NO_CLASS    0.063797
B           0.046027
J           0.069811
H           0.028038
Q           0.085989
M           0.077860
N           0.032003
C           0.050358
dtype: float64

In [ ]:
df_softmax["max_class_sim"] = [scores[i] for i in np.argmax(df_softmax[scores], 1)]
df_softmax["max_sim"] = df_softmax[scores].max(1)
(df_softmax.groupby("max_class_sim")["max_sim"].sum().sort_values(ascending=False)/len(df_softmax)).to_dict()

{'E': 0.050174398979562974,
 'A': 0.023796796839667064,
 'Q': 0.012052581654306763,
 'F': 0.00580021247840264,
 'M': 0.00459170423531266,
 'C': 0.0023795273779290737,
 'D': 0.0022758533080854447,
 'L': 0.0019484034991502789,
 'P': 0.0007996770780633897,
 'NO_CLASS': 0.0007033050583048011,
 'I': 0.00043587882957870895,
 'B': 0.00025407227715361147,
 'G': 0.00014115901784285303,
 'J': 0.00010460513368015456,
 'K': 0.00010311912013168188}